# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rslns/FlyRankAI_A1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### Rule

I will prioritize content items that show signs of being stale and have weak CTR relative to their search position. The score will combine `days_since_last_update` with a CTR-versus-position signal. Higher scores mean higher priority for review.

### Reason codes

* `STALE` — the content has not been updated recently.
* `CTR_POSITION` — CTR looks weak relative to the item's search position.
* `BOTH` — both signals indicate a potential issue.
* `NONE` — neither signal is strong enough for action.

The action label will be `REFRESH`, `REVIEW_CTR`, `REFRESH_AND_REVIEW`, or `NO_ACTION`.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the starter dataset for the baseline rule
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/rslns/FlyRankAI_A1/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))


Rows: 30000
Columns: 44


In [9]:
df["staleness_bucket"] = pd.qcut(
    df["days_since_last_update"],
    q=4,
    duplicates="drop"
)

stale_check = (
    df.groupby("staleness_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

print(stale_check)

  staleness_bucket      n  median_ctr  median_impressions
0    (0.999, 20.0]  15866        0.04               363.0
1    (20.0, 104.0]  13816        0.09              1262.0
2   (104.0, 373.0]    318        0.00                30.0


In [10]:
position_check = (
    df.groupby("position_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          median_position=("avg_position", "median")
      )
      .reset_index()
)

print(position_check)

  position_tier      n  median_ctr  median_position
0          deep   1319        0.00             61.0
1        page_1  11814        0.16              6.6
2      page_3_5   7242        0.03             28.9
3      striking   7304        0.11             13.9
4         top_3   2321        0.00              0.0


### Signal verdicts

**Staleness — MIXED.** Older content has lower median CTR in the oldest bucket, but the relationship is not monotonic and the oldest bucket contains only 318 rows. I will use staleness as a directional prioritization signal, not as proof that stale content performs worse.

**CTR versus position — MIXED.** CTR varies across position buckets, but the relationship is not clean. The `top_3` bucket also contains `avg_position = 0` values, which represent missing position data rather than actual position zero. I will therefore use this signal cautiously for prioritization.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
### Baseline rule

I will create a simple decision-support score using two observed signals: staleness and CTR relative to search position.

A higher score means higher priority for human review. The score is a baseline, not a prediction of future decline.

The rule will assign one reason code:

* `STALE` — staleness contributes the strongest signal.
* `CTR_POSITION` — weak CTR relative to search position contributes the strongest signal.
* `BOTH` — both signals are strong.
* `NONE` — neither signal is strong enough for action.

The action will be `REFRESH`, `REVIEW_CTR`, `REFRESH_AND_REVIEW`, or `NO_ACTION`.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Make a copy so the original dataframe stays unchanged
baseline = df.copy()

# Do not use avg_position == 0 as a real search position.
valid_position = baseline["avg_position"].replace(0, np.nan)

# Simple, transparent thresholds based on the observed data.
# Stale = more than 104 days since update.
# Weak CTR = below the median CTR for items with valid position data.
stale_flag = baseline["days_since_last_update"] > 104

ctr_threshold = baseline.loc[valid_position.notna(), "ctr"].median()
weak_ctr_flag = (
    valid_position.notna()
    & (baseline["ctr"] < ctr_threshold)
)

# Score: 1 point per signal.
baseline["score"] = (
    stale_flag.astype(int)
    + weak_ctr_flag.astype(int)
)

# One reason code
baseline["reason_code"] = np.select(
    [
        stale_flag & weak_ctr_flag,
        stale_flag,
        weak_ctr_flag
    ],
    [
        "BOTH",
        "STALE",
        "CTR_POSITION"
    ],
    default="NONE"
)

# Action label
baseline["action"] = np.select(
    [
        baseline["reason_code"] == "BOTH",
        baseline["reason_code"] == "STALE",
        baseline["reason_code"] == "CTR_POSITION"
    ],
    [
        "REFRESH_AND_REVIEW",
        "REFRESH",
        "REVIEW_CTR"
    ],
    default="NO_ACTION"
)

# Rank highest priority first
baseline = baseline.sort_values(
    ["score", "days_since_last_update"],
    ascending=[False, False]
).reset_index(drop=True)

# Save the required output
import os

os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

baseline[output_columns].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "work/outputs/baseline_action_score.csv")
print("Rows:", len(baseline))
print("CTR threshold:", ctr_threshold)

baseline[output_columns].head(10)

Saved: work/outputs/baseline_action_score.csv
Rows: 30000
CTR threshold: 0.08


,content_id,client_id,score,reason_code,action,days_since_last_update,ctr,avg_position,impressions_90d
0,content_55a5b1c46474,client_4ec9599fc2,2,BOTH,REFRESH_AND_REVIEW,373,0.0,7.5,35
1,content_f6fdf87348f6,client_4ec9599fc2,2,BOTH,REFRESH_AND_REVIEW,373,0.0,32.5,2
2,content_8d56efff1e71,client_4ec9599fc2,2,BOTH,REFRESH_AND_REVIEW,372,0.0,35.0,1
3,content_1b4ec72dafd4,client_4ec9599fc2,2,BOTH,REFRESH_AND_REVIEW,372,0.0,7.0,2
4,content_e2b702f4f92b,client_4ec9599fc2,2,BOTH,REFRESH_AND_REVIEW,334,0.0,9.3,30
5,content_06e19c6486b0,client_4ec9599fc2,2,BOTH,REFRESH_AND_REVIEW,334,0.0,5.0,10
6,content_7a888d3d99c8,client_19581e27de,2,BOTH,REFRESH_AND_REVIEW,313,0.0,67.6,95
7,content_6476d1d8c050,client_19581e27de,2,BOTH,REFRESH_AND_REVIEW,313,0.0,67.8,304
8,content_94991fe6268c,client_19581e27de,2,BOTH,REFRESH_AND_REVIEW,313,0.0,12.4,7
9,content_02b0d6e30129,client_19581e27de,2,BOTH,REFRESH_AND_REVIEW,313,0.0,6.9,176


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
### Top-20 review

I reviewed the highest-ranked items as decision-support recommendations rather than confirmed problems. The rule prioritizes items using observed staleness and CTR signals. Each recommendation can still be wrong because the signals may be incomplete, the content may have recovered, or low traffic may make the observed metrics unstable.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the actual top 20 picks for review

top20 = baseline[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "impressions_90d"
    ]
].head(20).copy()

print("Top 20 baseline picks:")
display(top20)

Top 20 baseline picks:


,content_id,score,reason_code,action,days_since_last_update,ctr,avg_position,impressions_90d
0,content_55a5b1c46474,2,BOTH,REFRESH_AND_REVIEW,373,0.0,7.5,35
1,content_f6fdf87348f6,2,BOTH,REFRESH_AND_REVIEW,373,0.0,32.5,2
2,content_8d56efff1e71,2,BOTH,REFRESH_AND_REVIEW,372,0.0,35.0,1
3,content_1b4ec72dafd4,2,BOTH,REFRESH_AND_REVIEW,372,0.0,7.0,2
4,content_e2b702f4f92b,2,BOTH,REFRESH_AND_REVIEW,334,0.0,9.3,30
5,content_06e19c6486b0,2,BOTH,REFRESH_AND_REVIEW,334,0.0,5.0,10
6,content_7a888d3d99c8,2,BOTH,REFRESH_AND_REVIEW,313,0.0,67.6,95
7,content_6476d1d8c050,2,BOTH,REFRESH_AND_REVIEW,313,0.0,67.8,304
8,content_94991fe6268c,2,BOTH,REFRESH_AND_REVIEW,313,0.0,12.4,7
9,content_02b0d6e30129,2,BOTH,REFRESH_AND_REVIEW,313,0.0,6.9,176


### Top-20 review

1. **content_55a5b1c46474** — Action: `REFRESH_AND_REVIEW`; reason: very stale (373 days) and CTR is 0.0. Confidence: low because it has only 35 impressions. It could be wrong because the sample is small and the page may not need a refresh.

2. **content_f6fdf87348f6** — Action: `REFRESH_AND_REVIEW`; reason: very stale (373 days) and CTR is 0.0. Confidence: low because only 2 impressions were recorded. It could be wrong because the CTR is based on almost no traffic.

3. **content_8d56efff1e71** — Action: `REFRESH_AND_REVIEW`; reason: 372 days stale and CTR is 0.0. Confidence: very low because there was only 1 impression. It could be wrong because the observed CTR is too sparse to be reliable.

4. **content_1b4ec72dafd4** — Action: `REFRESH_AND_REVIEW`; reason: 372 days stale and CTR is 0.0. Confidence: low because there were only 2 impressions. It could be wrong because the low impression count makes CTR unstable.

5. **content_e2b702f4f92b** — Action: `REFRESH_AND_REVIEW`; reason: 334 days stale and CTR is 0.0. Confidence: low because there were 30 impressions. It could be wrong because the observed CTR may not represent normal performance.

6. **content_06e19c6486b0** — Action: `REFRESH_AND_REVIEW`; reason: 334 days stale and CTR is 0.0. Confidence: low because there were only 10 impressions. It could be wrong because the sample is small.

7. **content_7a888d3d99c8** — Action: `REFRESH_AND_REVIEW`; reason: 313 days stale and CTR is 0.0. Confidence: moderate because it has 95 impressions. It could be wrong because zero clicks may still reflect limited traffic rather than a content problem.

8. **content_6476d1d8c050** — Action: `REFRESH_AND_REVIEW`; reason: 313 days stale and CTR is 0.0. Confidence: moderate because it has 304 impressions. It could be wrong because CTR alone does not establish that updating the content will improve performance.

9. **content_94991fe6268c** — Action: `REFRESH_AND_REVIEW`; reason: 313 days stale and CTR is 0.0. Confidence: very low because there were only 7 impressions. It could be wrong because the CTR observation is extremely sparse.

10. **content_02b0d6e30129** — Action: `REFRESH_AND_REVIEW`; reason: 313 days stale and CTR is 0.0. Confidence: moderate because it has 176 impressions. It could be wrong because the rule does not consider content quality or search intent.

11. **content_f488400fca67** — Action: `REFRESH_AND_REVIEW`; reason: 305 days stale and CTR is 0.0. Confidence: moderate because it has 155 impressions. It could be wrong because the rule cannot establish that staleness caused the low CTR.

12. **content_026a1e2a82fd** — Action: `REFRESH_AND_REVIEW`; reason: 305 days stale and CTR is 0.0. Confidence: low because it has only 13 impressions. It could be wrong because the traffic sample is small.

13. **content_d25a099b3726** — Action: `REFRESH_AND_REVIEW`; reason: 305 days stale and CTR is 0.0. Confidence: moderate because it has 202 impressions. It could be wrong because a refresh may not address the actual reason for zero clicks.

14. **content_129753e3095f** — Action: `REFRESH_AND_REVIEW`; reason: 305 days stale and CTR is 0.0. Confidence: very low because it has only 7 impressions. It could be wrong because the CTR signal is too sparse.

15. **content_f2b4acf220d9** — Action: `REFRESH_AND_REVIEW`; reason: 305 days stale and CTR is 0.0. Confidence: low because it has only 17 impressions. It could be wrong because the observed CTR may be unstable.

16. **content_afd26a07382d** — Action: `REFRESH_AND_REVIEW`; reason: 305 days stale and CTR is 0.0. Confidence: low because it has 15 impressions. It could be wrong because the rule may over-prioritize low-traffic pages.

17. **content_ab18b5811c02** — Action: `REFRESH_AND_REVIEW`; reason: 305 days stale and CTR is 0.0. Confidence: very low because it has only 5 impressions. It could be wrong because there is insufficient traffic to judge CTR reliably.

18. **content_57d8360dee59** — Action: `REFRESH_AND_REVIEW`; reason: 305 days stale and CTR is 0.0. Confidence: low because it has 10 impressions. It could be wrong because the low CTR may simply reflect limited observations.

19. **content_dd413158df3c** — Action: `REFRESH_AND_REVIEW`; reason: 305 days stale and CTR is 0.0. Confidence: low because it has 13 impressions. It could be wrong because the sample is too small to strongly support the action.

20. **content_ab27c30d81f4** — Action: `REFRESH_AND_REVIEW`; reason: 304 days stale and CTR is 0.0. Confidence: moderate because it has 103 impressions. It could be wrong because the baseline does not account for other reasons a page may have low CTR.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
### Weak picks

Several top-ranked picks look weak despite receiving the maximum score. The clearest examples are `content_8d56efff1e71` with only 1 impression, `content_f6fdf87348f6` with 2 impressions, and `content_ab18b5811c02` with 5 impressions. Their zero CTR is not strong evidence because the number of observations is extremely small.

The baseline also tends to group many pages with the same score, so the ordering within the top group is not very informative. This is a limitation of the simple rule.

### Leakage check

The baseline uses only current/observed fields: `days_since_last_update`, `ctr`, and `avg_position` for the rule. It does not use `trend_pct`, `trend_direction`, `is_declining_label`, or any future-window outcome. Client and content IDs are retained only for identification and output, not as model signals.

This is a directional decision-support baseline, not a claim that these pages will decline or that refreshing them will definitely improve performance.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Explicit leakage check

leakage_columns = [
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

present_leakage = [
    col for col in leakage_columns
    if col in baseline.columns
]

rule_features = [
    "days_since_last_update",
    "ctr",
    "avg_position"
]

print("Label-derived columns present in rule features:",
      [col for col in present_leakage if col in rule_features])

print("Rule features used:", rule_features)

assert "trend_pct" not in rule_features
assert "trend_direction" not in rule_features
assert "is_declining_label" not in rule_features

print("Leakage check: PASSED")

Label-derived columns present in rule features: []
Rule features used: ['days_since_last_update', 'ctr', 'avg_position']
Leakage check: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.